# Volume benchmarking with DoMINO NIM (DrivAerML)

This notebook lives under **`workflows/nim_inference/notebooks/`**. It calls **`physicsnemo.cfd.evaluation.nims.call_domino_nim`** against a running [DoMINO Automotive Aerodynamics NIM](https://catalog.ngc.nvidia.com/orgs/nim/teams/nvidia/containers/domino-automotive-aero), maps predictions onto DrivAerML volume meshes, and evaluates metrics via **`physicsnemo.cfd.postprocessing_tools`**. For Hydra matrix benchmarks with packaged models, use **[`workflows/benchmarking/`](../../benchmarking/README.md)**.

We compare NIM predictions with ground truth for external aerodynamics using volume fields from [DrivAerML](https://huggingface.co/datasets/neashton/drivaerml/) (sample ID **202**). Volume VTU files are large; ensure sufficient memory and disk.

## Prerequisites

Install **PhysicsNeMo-CFD** from the repository root (editable install recommended). With GPU extras:

```bash
cd physicsnemo-cfd   # repository root
pip install -e ".[gpu]" --extra-index-url=https://pypi.nvidia.com
```

Run the [DoMINO Automotive Aerodynamics NIM](https://docs.nvidia.com/nim/physicsnemo/domino-automotive-aero/latest/quickstart-guide.html) so `call_domino_nim` can reach `http://localhost:8000/v1/infer` (or set your URL). If this notebook runs in Docker next to the NIM, use **`--network host`** on both containers.


## Compute model inference on the benchmark simulation

The benchmark results are saved in `.vtp` (surface) and `.vtu` (volume) formats. For this notebook, let's start by loading the `.vtu` file and inspecting the available fields. We will use the `pyvista` library for this purpose.

---
**NOTE**

The volume files in the DrivAerML dataset are large (each file is ~38 GB). Ensure you have sufficient memory to load them.

---

### Download the benchmark solution

Let's download the solution from the [DrivAerML dataset](https://huggingface.co/datasets/neashton/drivaerml).

In [ ]:
import os

filenames = [
    "drivaer_202.stl",
    "volume_202.vtu.00.part",
    "volume_202.vtu.01.part",
]
urls = [
    "https://huggingface.co/datasets/neashton/drivaerml/resolve/main/run_202/drivaer_202.stl",
    "https://huggingface.co/datasets/neashton/drivaerml/resolve/main/run_202/volume_202.vtu.00.part",
    "https://huggingface.co/datasets/neashton/drivaerml/resolve/main/run_202/volume_202.vtu.01.part",
]

for filename, url in zip(filenames, urls):
    if not os.path.exists(filename):
        !wget "{url}" -O "{filename}"
    else:
        print(f"{filename} already exists. Skipping download.")

if not os.path.exists("volume_202.vtu"):
    !cat "volume_202.vtu.00.part" "volume_202.vtu.01.part" > "volume_202.vtu"

In [ ]:
import pyvista as pv

mesh = pv.read("./volume_202.vtu")
mesh

We observe that the mesh contains `165,122,607` points and `146,754,704` cells. The pressure, velocity, and turbulent viscosity variables are stored as `pMeanTrim`, `UMeanTrim`, and `nutMeanTrim`, respectively. We'll denote the model's predictions as `pMeanTrimPred`, `UMeanTrimPred`, and `nutMeanTrimPred`. We compute the inference using the [DoMINO Automotive-Aero NIM](https://docs.nvidia.com/nim/physicsnemo/domino-automotive-aero/latest/overview.html), but you can adapt the code to your specific model inputs/outputs. Unlike the surface notebook, we will perform all computations on point data, and we will store model results on point data as well.

---
**NOTE**

The DoMINO NIM provides flow fields inside a specific bounding box surrounding the car. Specifically, the bounding box extends from -3.5 to 8.5 in the x-direction, -2.25 to 2.25 in the y-direction, and -0.32 to 3.00 in the z-direction. Outside this bounding box, we replace the model predictions with zeros.

---

---
**NOTE**

If you are using a Docker container to run this notebook (e.g., the [PhysicsNeMo Docker container](https://catalog.ngc.nvidia.com/orgs/nvidia/teams/physicsnemo/containers/physicsnemo)), ensure that you start both the NIM container and the container running this notebook with the `--network host` flag so they can communicate.

---

---
**IMPORTANT**

We are using the DrivAerML dataset with the DoMINO NIM for this example. The DoMINO NIM is trained on RANS data, whereas the DrivAerML dataset was generated using RANS-LES (HRLES) simulations. Consequently, the DoMINO NIM may show inaccuracies in turbulent quantities (e.g., turbulent viscosity and wall shear stress) relative to the ground-truth (HRLES) data. To see the true performance of the DoMINO model architecture trained on the DrivAerML dataset, refer to [Using standardized datasets for inter-model comparisons](../../deprecated/bench_example/README.md#using-standardized-datasets-for-inter-model-comparisons) in the deprecated bench_example README.

---

In [ ]:
import numpy as np
import os, time
from physicsnemo.cfd.postprocessing_tools.interpolation.interpolate_mesh_to_pc import (
    _create_nbrs_surface,
    _interpolate,
)
from physicsnemo.cfd.evaluation.nims import call_domino_nim

output_dict = call_domino_nim(
    stl_path="./drivaer_202.stl",
    inference_api_url="http://localhost:8000/v1/infer",
    data={
        "stream_velocity": "38.89",
        "stencil_size": "1",
        "point_cloud_size": "500000",
    },
    verbose=True,
)
print("Interpolating results on the benchmarking mesh...")

start_time = time.time()
# Interpolate the results of the NIM onto the mesh points
# v2 of NIM supports inference on custom point clouds. 
# you can also pass a custom point cloud directly, however for backwards
# compatability, we will continue the interpolation method. 
# If you wish to infer directly on the point cloud, please uncomment the code
# below and make appropirate changes
# pc_path = "volume_202_pc.npy"
# if not os.path.exists(pc_path):
#     np.save(pc_path, mesh.points)
# 
# output_dict = call_domino_nim(
#     stl_path="./drivaer_202.stl",
#     inference_api_url="http://localhost:8000/v1/infer",
#     data={
#         "stream_velocity": "38.89",
#         "stencil_size": "1",
#     },
#     batch_size=32000,
#     point_cloud="./volume_202_pc.npy",
#     verbose=True,
#     timeout=300,
# )

nbrs_surface = _create_nbrs_surface(
    output_dict["coordinates"][0, :], n_neighbors=3, device="gpu"
)
fields = np.concatenate(
    [
        output_dict["pressure"][0, :],
        output_dict["velocity"][0, :],
        output_dict["turbulent_viscosity"][0, :],
    ],
    axis=1,
)
fields_interp = _interpolate(
    nbrs_surface, mesh.points, fields, device="gpu", batch_size=10_000_000
)
mesh.point_data["pMeanTrimPred"] = fields_interp[:, 0]
mesh.point_data["UMeanTrimPred"] = fields_interp[:, 1:4]
mesh.point_data["nutMeanTrimPred"] = fields_interp[:, 4]
print(f"Interpolation took {time.time() - start_time:.3f} sec")

points = mesh.points
bounds = [-3.5, 8.5, -2.25, 2.25, -0.32, 3.00]
mask = (
    (points[:, 0] >= bounds[0])
    & (points[:, 0] <= bounds[1])
    & (points[:, 1] >= bounds[2])
    & (points[:, 1] <= bounds[3])
    & (points[:, 2] >= bounds[4])
    & (points[:, 2] <= bounds[5])
)

mesh.point_data["pMeanTrimPred"][~mask] = 0.0
mesh.point_data["UMeanTrimPred"][~mask] = 0.0
mesh.point_data["nutMeanTrimPred"][~mask] = 0.0

mesh

Now that we have both the predicted and true fields on the same points, we can start comparing these two solutions.

## L2 Errors

Let's compare the L2 errors for the pressure, velocity, and turbulent viscosity fields using the library function `compute_l2_errors`. We pass the `bounds` argument to restrict the computation to the region where model results are available.

In [ ]:
from physicsnemo.cfd.postprocessing_tools.metrics.l2_errors import compute_l2_errors

l2_errors = compute_l2_errors(
    mesh,
    true_fields=["pMeanTrim", "UMeanTrim"],
    pred_fields=["pMeanTrimPred", "UMeanTrimPred"],
    bounds=[-3.5, 8.5, -2.25, 2.25, -0.32, 3.00],
    dtype="point",
)
l2_errors

L2 errors provide an aggregate sense of model performance, but they do not reveal the distribution of errors. Below, we compute the distribution of absolute errors as a function of distance from the STL.

In [ ]:
from physicsnemo.cfd.postprocessing_tools.metrics.l2_errors import compute_error_vs_sdf
import matplotlib.pyplot as plt

# determine query points for sdf
bin_edges = np.linspace(0, 2, 50, endpoint=False)
bin_edges = np.concatenate([bin_edges, np.linspace(2, 8, 51)])

stl_mesh = pv.read("./drivaer_202_single_solid.stl")

output_dict = compute_error_vs_sdf(
    mesh,
    ["pMeanTrim", "UMeanTrim", "nutMeanTrim"],
    ["pMeanTrimPred", "UMeanTrimPred", "nutMeanTrimPred"],
    stl_mesh,
    bin_edges,
    bounds=[-3.5, 8.5, -2.25, 2.25, -0.32, 3.00],
    dtype="point"
)
# print(output_dict)
keys_to_plot = [key for key in output_dict.keys() if "histogram" in key]
fig, axs = plt.subplots(1, len(keys_to_plot), figsize=(18,6))
for i, key in enumerate(keys_to_plot):
    ax = axs[i]
    bin_centers = output_dict[key]["bin_edges"]
    bin_centers = [(a + b) / 2 for a, b in zip(bin_centers, bin_centers[1:])]
    errors = output_dict[key]["mean_errors"]
    ax.plot(bin_centers, errors, marker='o')
    ax.set_title(key)
    ax.set_xlabel('Distance from Surface of Vehicle')
    ax.set_ylabel('Mean Error per point')
    ax.grid(True, alpha=0.3)
plt.tight_layout()

We observe that for pressure and velocity, error is higher near the STL surface, while for turbulent viscosity the trend is reversed. This can inform model architecture or loss weighting choices. Below, we show this distribution computed across the entire DrivAerML validation dataset.

![Aggregate Error as a function of SDF](img/error_vs_sdf.png)


## Plotting fields

Much of volumetric data analysis relies on visualizing the flow field for specific features of interest. Let's create a few slices and use `plot_field_comparisons` to generate visuals.


In [ ]:
# Create a slice that runs along the centerline
y_slice = mesh.slice(normal="y", origin=(0, 0, 0))
y_slice = y_slice.clip_box(bounds, invert=False)

# Create a slice to visualize the wake of the car
x_4_slice = mesh.slice(normal="x", origin=(4, 0, 0))
x_4_slice = x_4_slice.clip_box(bounds, invert=False)

# Create a slice normal to z axis to visualize the wheel wakes
z_neg_0_2376_slice = mesh.slice(normal="z", origin=(0, 0, -0.2376))
z_neg_0_2376_slice = z_neg_0_2376_slice.clip_box(bounds, invert=False)

In [ ]:
from physicsnemo.cfd.postprocessing_tools.visualization.utils import plot_field_comparisons

_xvfb = getattr(pv, "start_xvfb", None)
if _xvfb is not None:
    _xvfb()
else:
    pv.OFF_SCREEN = True

plotter = plot_field_comparisons(
    y_slice,
    true_fields=["pMeanTrim", "UMeanTrim"],
    pred_fields=["pMeanTrimPred", "UMeanTrimPred"],
    plot_vector_components=False,
    view="xz",
    dtype="point",
    cmap="jet",
    lut=20,
    window_size=[2560, 1280],
)

plotter.screenshot("./sample_202_volume_y_slice_comparison.png")

# Display the image
from IPython.display import Image

Image(filename="./sample_202_volume_y_slice_comparison.png")

In [ ]:
plotter = plot_field_comparisons(
    x_4_slice,
    true_fields=["pMeanTrim", "UMeanTrim"],
    pred_fields=["pMeanTrimPred", "UMeanTrimPred"],
    plot_vector_components=False,
    view="yz",
    dtype="point",
    cmap="jet",
    lut=20,
    window_size=[2560, 1280],
)

plotter.screenshot("./sample_202_volume_x_4_slice_comparison.png")

# Display the image
from IPython.display import Image

Image(filename="./sample_202_volume_x_4_slice_comparison.png")

In [ ]:
plotter = plot_field_comparisons(
    z_neg_0_2376_slice,
    true_fields=["pMeanTrim", "UMeanTrim"],
    pred_fields=["pMeanTrimPred", "UMeanTrimPred"],
    plot_vector_components=False,
    view="xy",
    dtype="point",
    cmap="jet",
    lut=20,
    window_size=[2560, 1280],
)

plotter.screenshot("./sample_202_volume_z_slice_comparison.png")

# Display the image
from IPython.display import Image

Image(filename="./sample_202_volume_z_slice_comparison.png")

Slice visualizations show that the AI prediction captures the larger structures in the flow field. Predictions are somewhat noisy, some of which can be attributed to interpolating NIM results from ~500k points to ~160M points.

Such visualizations do not reveal how predictions look across the entire validation dataset. For that, one can project errors from different samples onto fixed points and build aggregate visualizations. The [Hydra benchmarking workflow](../../benchmarking/README.md) provides aggregate visualizations across many cases.

Below we show the error distribution across the entire DrivAerML validation set (visualized using sample ID 439, the largest STL by geometric size).

![Aggregate Errors](img/resampled_volume_errors.png)

Overall, errors are typically higher in the wake for velocity, closer to the car surface for pressure, and in the farfield for turbulent viscosity.

Such analysis is especially useful when geometric differences between samples are modest (e.g., DrivAerML), where resampling techniques enable meaningful aggregate visualizations.

Let's also create line plots for more detailed visualization. Here, we visualize the wake behind the wheels and the flow along the centerline under the car, similar to the results presented in the [DrivAerML paper](https://arxiv.org/abs/2408.11969). 

In [ ]:
from physicsnemo.cfd.postprocessing_tools.visualization.utils import plot_line

centerline_bottom = y_slice.slice(normal="z", origin=(0, 0, -0.2376))

fig = plot_line(
    centerline_bottom,
    plot_coord="x",
    field_true="UMeanTrim",
    field_pred="UMeanTrimPred",
    normalize_factor=38.889,
    coord_trim=(-1.0, 6.0),
    field_trim=(0, 2.0),
    flip=False,
    true_line_kwargs={"color": "red", "label": "True"},
    pred_line_kwargs={"color": "green", "label": "Pred"},
    xlabel="X Coordinate",
    ylabel="U / U_ref",
)

In [ ]:
front_wheel_wake = z_neg_0_2376_slice.slice(normal="x", origin=(0.35, 0, 0))

fig = plot_line(
    front_wheel_wake,
    plot_coord="y",
    field_true="UMeanTrim",
    field_pred="UMeanTrimPred",
    normalize_factor=38.889,
    coord_trim=(-1.0, 1.0),
    field_trim=(0, 2.0),
    flip=False,
    true_line_kwargs={"color": "red", "label": "True"},
    pred_line_kwargs={"color": "green", "label": "Pred"},
    xlabel="Y Coordinate",
    ylabel="U / U_ref",
)

In [ ]:
rear_wheel_wake = z_neg_0_2376_slice.slice(normal="x", origin=(3.15, 0, 0))

fig = plot_line(
    rear_wheel_wake,
    plot_coord="y",
    field_true="UMeanTrim",
    field_pred="UMeanTrimPred",
    normalize_factor=38.889,
    coord_trim=(-1.0, 1.0),
    field_trim=(0, 2.0),
    flip=False,
    true_line_kwargs={"color": "red", "label": "True"},
    pred_line_kwargs={"color": "green", "label": "Pred"},
    xlabel="Y Coordinate",
    ylabel="U / U_ref",
)

Generally, the line plots show good correlation between the AI model results and the ground truth. However, the model struggles to capture wake behavior.

That completes the volume benchmarking notebook. For surface benchmarking, see [`surface_benchmarking.ipynb`](./surface_benchmarking.ipynb). To run these metrics across multiple geometries or cases, use the [Hydra benchmarking workflow](../../benchmarking/README.md) (`python main.py` from `workflows/benchmarking/`).

## Bonus: Computing the Equation Residuals

Continuity and the Momentum Equations (Navier-Stokes) are the fundamental equations that govern the fluid dynamics of the external aero. We can use the library to measure how well the model's results capture the mass and energy balances. We can use the `compute_continuity_residuals` and `compute_momentum_residuals` functions for these.

---
**NOTE**

Due the the large size of the mesh, this computation can take a few minutes. If you are not interested in the this metric, you can skip the below code blocks. 

---

In [ ]:
from physicsnemo.cfd.postprocessing_tools.metrics.physics import (
    compute_continuity_residuals,
    compute_momentum_residuals,
)

# Let's clip the mesh to work on a smaller dataset
# Set crinkle=True for faster clipping
clipped_mesh = mesh.clip_box(
    bounds=[-2, 6, -0.20, 0.20, -0.32, 3.00],
    invert=False,
    progress_bar=True,
    merge_points=False,
    crinkle=True,
)

# Compute continuity
clipped_mesh = compute_continuity_residuals(
    clipped_mesh,
    true_velocity_field="UMeanTrim",
    predicted_velocity_field="UMeanTrimPred",
)

# Optionally, compute momentum using RANS equations
nu = 1.507e-5
rho = 1.0
clipped_mesh = compute_momentum_residuals(
    clipped_mesh,
    true_velocity_field="UMeanTrim",
    predicted_velocity_field="UMeanTrimPred",
    true_pressure_field="pMeanTrim",
    predicted_pressure_field="pMeanTrimPred",
    true_nu_field="nutMeanTrim",
    predicted_nu_field="nutMeanTrimPred",
    nu=nu,
    rho=rho,
)

Note that the above two functions will create additional fields for `Continuity`, `ContinuityPred`, `Momentum` and `MomentumPred`. The resulting mesh files can be processed as before. For the purposes of this notebook, let's plot the the fields along the mid-y slice. 

In [ ]:
y_slice = clipped_mesh.slice(normal="y", origin=(0, 0, 0))
y_slice = y_slice.clip_box(bounds, invert=False)

plotter = plot_field_comparisons(
    y_slice,
    true_fields=["Continuity", "Momentum"],
    pred_fields=["ContinuityPred", "MomentumPred"],
    plot_vector_components=True,
    view="xz",
    dtype="point",
    cmap="jet",
    lut=20,
    window_size=[2560, 2560],
)

plotter.screenshot("./sample_202_volume_y_slice_residuals_comparison.png")

# Display the image
from IPython.display import Image

Image(filename="./sample_202_volume_y_slice_residuals_comparison.png")

The predicted residuals show good adherence of the model predictions to the governing laws. The residuals are generally seen to be higher in the wake area or the areas of high shear.

We can also compute the residuals in an integral sense. For example, below code demonstrates integral continuity computed on a box surrounding the car.

In [ ]:
from physicsnemo.metrics.cae.integral import surface_integral
from physicsnemo.cfd.postprocessing_tools.interpolation.interpolate_mesh_to_pc import interpolate_mesh_to_pc

# Define a box smaller than the overall bounding box
integral_box_bounds = [-2, 5, -1.5, 1.5, -0.2, 1.5]
integral_box = pv.Box(integral_box_bounds, level=40)
integral_box = interpolate_mesh_to_pc(integral_box, mesh, ["UMeanTrim", "UMeanTrimPred"], mesh_dtype="point")

integrals = surface_integral(integral_box)
integrals

 We can observe that the integral continuity (integral U) for the predicted result is further away from 0 compared to the true result. It is quite interesting however to note that even the true solution does not respect the continuity perfectly. 